# Implement ROPE from Scratch
# 🔁 Rotary Positional Embeddings in PyTorch

### 🧠 Problem Statement
Transformers need a sense of **order**, but vanilla attention mechanisms are position-agnostic. Positional encodings help inject this order-awareness into the model. 

Your mission is to implement **Rotary Positional Embeddings (RoPE)** from scratch — a newer and slicker technique that rotates the query and key vectors instead of simply adding sine-cosine vectors. This method preserves attention efficiency while enabling better generalization for long sequences.

---

### ✅ Requirements

1. **Implement the Rotary Module**
   - Construct a `Rotary` class to compute sinusoidal frequencies.
   - Precompute and cache `cos` and `sin` values per sequence length.
   - Register these as buffers to keep them on the correct device.

2. **Define Rotation Helpers**
   - `rotate_half(x)` splits and rotates half the dimensions of a tensor.
   - `apply_rotary_pos_emb(q, k, cos, sin)` applies these rotations to Q and K.

3. **Simulate Usage**
   - Create synthetic tensors for Q, K, V.
   - Generate rotary embeddings using the custom `Rotary` module.
   - Apply rotary embeddings to Q and K.

4. **Verify Dimensions**
   - Final shapes should align with expected shapes for attention modules.
   - Confirm RoPE is applied before dot-product attention would normally occur.

---

### 📏 Constraints

- ✅ Use only PyTorch — no Fairseq or HuggingFace positional modules.
- ✅ Must support dynamic sequence lengths and cache embeddings per sequence.
- ✅ Should handle odd/even dimensional splits correctly.
- ❌ Do **not** manually plug in Fairseq’s `SinusoidalPositionalEmbedding`.

---

<details>
  <summary>💡 Hint</summary>
  - Use `torch.einsum("i,j->ij", t, inv_freq)` to compute frequency pairs.
  - Cache the cosine and sine values in the `Rotary` class using `self.register_buffer()`.
  - The `rotate_half(x)` function should split `x` into two halves and rotate them: `[-x2, x1]`.
  - Apply the rotary transformation using:  
    `(q * cos) + (rotate_half(q) * sin)`  
    and similarly for `k`.
  - Remember to broadcast `cos` and `sin` to match the shape of `q` and `k`.
</details>

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math

In [21]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

torch.Size([3, 4, 8])


In [28]:
class Rotary(torch.nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        # [D / 2]
        inv_freq = 1 / 10000 ** (torch.arange(0, d_model, 2).float() / d_model)
        
        # [T]
        t = torch.arange(0, max_seq_len).type_as(inv_freq)
        # [T, D/2]
        pairs = torch.einsum('i,j->ij', t, inv_freq)
        emb = torch.cat([pairs, pairs], dim=-1)
        sin_cache = emb.sin()
        cos_cache = emb.cos()
        self.register_buffer('sin_cache', sin_cache)
        self.register_buffer('cos_cache', cos_cache)

    def forward(self, seq_len: int):
        return self.sin_cache[:seq_len], self.cos_cache[:seq_len]

def rotate_half(x: torch.Tensor):
    half_d = d_model // 2
    x1, x2 = x[..., : half_d], x[..., half_d :]
    x = torch.cat([-x2, x1], dim=-1)
    return x

# @torch.jit.script
def apply_rotary_pos_emb(q, k, sin, cos):
    return cos * q + rotate_half(q) * sin, cos * k + rotate_half(k)

In [29]:
# from fairseq.modules.sinusoidal_positional_embedding import SinusoidalPositionalEmbedding

max_seq_len = 100
# d_model = 64

# Fairseq's implementation requires the number of embeddings (seq length) and embedding dim
# pos_emb = SinusoidalPositionalEmbedding(d_model, max_seq_len, padding_idx=None)

# Generate embeddings for a sequence of length 50

rotary = Rotary(d_model, max_seq_len)
sin, cos = rotary(seq_len)

q, k = apply_rotary_pos_emb(q, k, sin, cos)

print(q.shape)  # (1, 50, 64)


torch.Size([3, 4, 8])
